# Урок 29 — SQL: Від Синтаксису до Архітектури

> **Інструмент:** вбудована бібліотека Python `sqlite3` (встановлення не потрібне)

---

## Що ми вивчимо

| Розділ | Тема |
|--------|------|
| 1 | SQL — це не просто мова запитів |
| 2 | 5 підмов SQL: DDL, DML, DQL, DCL, TCL |
| 3 | Практика з `sqlite3`: CREATE, INSERT, SELECT |
| 4 | Транзакції та ACID |
| 5 | Реляційна архітектура: зв'язки та JOIN |
| 6 | SQL як машина станів |


---
## 1. SQL — операційна система для даних

Назва **Structured Query Language** (структурована мова запитів) — це **історична помилка**.

Запити (`SELECT`) — лише одна з п'яти ролей SQL. Архітектурно SQL — це **система керування станом** (`state management system`).

### Чому "стан"?

Код вашого застосунку — **ephemeral** (короткочасний): перезапустіть сервер — і він забуде все.
Справжній "стан бізнесу" (хто заплатив, скільки товару на складі, хто авторизований) — **живе в базі даних**.

SQL — декларативний інтерфейс, що:
- визначає **фізику цього стану** (правила, обмеження, структуру)
- гарантує математично коректні **переходи між станами** (транзакції)
- захищає від одночасних конкурентних змін від багатьох користувачів

```
База даних = єдина велика змінна, значення якої змінюється в часі
              (колекція "аксіом" — фактів, які система вважає правдивими)
```

### Архітектурна метафора

Уявіть базу даних як **склад з суворими правилами**:

| Роль | Аналогія | SQL підмова |
|------|----------|-------------|
| Архітектор (будує приміщення) | Міський планувальник | **DDL** |
| Вантажники (кладуть товар) | Складські робітники | **DML** |
| Охоронна камера (спостерігає) | Аудитор | **DQL** |
| Охоронець (перевіряє доступ) | Бодігард + картки доступу | **DCL** |
| Нотаріус (гарантує угоди) | Ескроу-агент | **TCL** |


---
## 2. П'ять підмов SQL

### DDL — Data Definition Language (Архітектор)

**Роль:** визначає структуру бази даних — таблиці, обмеження, індекси. Працює зі **схемою**, не з даними.

**Команди:**
- `CREATE` — створити нову структуру (таблицю, представлення)
- `ALTER` — змінити існуючу структуру
- `DROP` — повністю видалити структуру **разом із даними**

⚠️ **Типова помилка:** плутати `DROP TABLE` (DDL — видаляє саму таблицю) з `DELETE FROM table` (DML — видаляє лише дані, структура залишається).

---

### DML — Data Manipulation Language (Оператор даних)

**Роль:** змінює **вміст** таблиць (не структуру). Відповідає за C, U, D в парадигмі CRUD.

**Команди:**
- `INSERT` — додати новий рядок
- `UPDATE` — змінити поля в існуючих рядках
- `DELETE` — видалити рядки

⚠️ **Типова помилка:** `UPDATE` або `DELETE` без `WHERE` — перезапише/видалить **всю** таблицю!

---

### DQL — Data Query Language (Спостерігач)

**Роль:** читає дані **не змінюючи** їх. Відповідає за R у CRUD.

**Команди:** `SELECT`

⚠️ **Типова помилка:** `SELECT *` — "забирає весь склад" замість конкретних колонок. Руйнує продуктивність на великих таблицях.

---

### DCL — Data Control Language (Охоронець)

**Роль:** керує **правами доступу** для користувачів і ролей.

**Команди:**
- `GRANT` — надати привілеї
- `REVOKE` — відібрати привілеї

⚠️ **Типова помилка:** надавати DML/DDL права аналітику, якому потрібен лише `SELECT`. Також — коли "таблиця не існує", часто це насправді DCL-проблема: таблиця є, але користувач не має прав.

> 📌 **SQLite не підтримує DCL** — це файлова БД без системи користувачів. У PostgreSQL/MySQL DCL є повноцінним.

---

### TCL — Transaction Control Language (Нотаріус)

**Роль:** групує DML-операції в **атомарні одиниці роботи** (транзакції). Або все — або нічого.

**Команди:**
- `BEGIN` / `START TRANSACTION` — почати відстежувати зміни
- `COMMIT` — зберегти зміни назавжди
- `ROLLBACK` — скасувати всі зміни з початку транзакції
- `SAVEPOINT` — позначка всередині транзакції для часткового відкату

⚠️ **Типова помилка:** забути зробити `COMMIT` — рядки залишаться заблокованими. Або не використовувати TCL взагалі, що призведе до часткових станів при збоях.


---
## 3. Практика: sqlite3

### Налаштування з'єднання

`sqlite3` — вбудована бібліотека Python. Не потребує сервера: база даних — це просто один файл (або `:memory:` для тимчасової бази в RAM).

In [ ]:
import sqlite3

# Створюємо базу даних в оперативній пам'яті
# ':memory:' — спеціальне ім'я, яке означає "тимчасова БД тільки в RAM"
# Для постійної БД вкажіть шлях до файлу: sqlite3.connect('mydb.db')
conn = sqlite3.connect(':memory:')

# cursor — об'єкт для виконання SQL-запитів
# Аналогія: cursor = рука, яка тримає ручку і пише команди в базу
cursor = conn.cursor()

# row_factory дозволяє звертатись до результатів як до словника: row['name']
# замість індексів: row[0]
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

print('З\'єднання з SQLite встановлено!')
print(f'Версія SQLite: {sqlite3.sqlite_version}')

### DDL на практиці — CREATE TABLE

In [ ]:
# ---- DDL: Визначаємо структуру бази даних ----
# Команда CREATE TABLE повністю описує схему: типи колонок, обмеження, ключі

# Таблиця відділів (departments) — батьківська сутність
cursor.execute('''
    CREATE TABLE IF NOT EXISTS departments (
        -- PRIMARY KEY: унікальний ідентифікатор рядка, SQLite додає AUTOINCREMENT автоматично
        dept_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        -- NOT NULL: ця колонка обов'язкова — порожнє значення заборонено
        dept_name TEXT    NOT NULL,
        -- UNIQUE: два відділи не можуть мати однакову назву
        location  TEXT    NOT NULL,
        budget    REAL    DEFAULT 0.0   -- DEFAULT: значення за замовчуванням
    )
''')

# Таблиця співробітників (employees) — дочірня сутність
cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        emp_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        first_name  TEXT    NOT NULL,
        last_name   TEXT    NOT NULL,
        salary      REAL    NOT NULL CHECK(salary > 0),  -- CHECK: власна бізнес-умова
        email       TEXT    UNIQUE,                       -- Унікальний email
        -- FOREIGN KEY: зв'язок з таблицею departments
        -- dept_id в employees = dept_id в departments (зв'язок 1:M)
        -- ON DELETE SET NULL: якщо відділ видалити — співробітник залишається без відділу (dept_id = NULL)
        dept_id     INTEGER REFERENCES departments(dept_id) ON DELETE SET NULL
    )
''')

# Зберігаємо зміни в базі
conn.commit()
print('Таблиці departments та employees створено!')

In [ ]:
# ---- DDL: ALTER TABLE — змінюємо існуючу структуру ----
# ALTER TABLE дозволяє додавати колонки без пересоздання таблиці
# ВАЖЛИВО: SQLite має обмежений ALTER TABLE — видалити колонку можна лише з v3.35.0+

cursor.execute('''
    ALTER TABLE employees
    ADD COLUMN hire_date TEXT DEFAULT '2024-01-01'
''')

conn.commit()
print('Колонку hire_date додано до таблиці employees!')

# Переглядаємо оновлену структуру таблиці
cursor.execute("PRAGMA table_info(employees)")
columns = cursor.fetchall()
print('\nСтруктура таблиці employees:')
print(f'{"ID":<5} {"Назва":<15} {"Тип":<10} {"Обов\'язк":<10} {"За замов.":<15} {"PK"}')
print('-' * 65)
for col in columns:
    print(f'{col[0]:<5} {col[1]:<15} {col[2]:<10} {col[3]:<10} {str(col[4]):<15} {col[5]}')

### DML на практиці — INSERT, UPDATE, DELETE

In [ ]:
# ---- DML: INSERT — додаємо дані в таблиці ----

# Спочатку заповнюємо батьківську таблицю departments
departments_data = [
    ('Розробка',      'Київ',   500000.0),
    ('Маркетинг',     'Харків', 200000.0),
    ('Аналітика',     'Львів',  300000.0),
    ('HR',            'Київ',   150000.0),
]

# executemany — ефективне масове вставлення (один раунд замість N)
# Параметри '?' — захист від SQL injection (ніколи не форматуйте рядки вручну!)
cursor.executemany(
    'INSERT INTO departments (dept_name, location, budget) VALUES (?, ?, ?)',
    departments_data
)

# Тепер заповнюємо дочірню таблицю employees
employees_data = [
    ('Олег',    'Коваль',   85000.0,  'o.koval@company.ua',   1, '2022-03-15'),
    ('Марія',   'Шевченко', 92000.0,  'm.shevchenko@co.ua',   1, '2021-07-01'),
    ('Дмитро',  'Бондар',   70000.0,  'd.bondar@company.ua',  2, '2023-01-10'),
    ('Анна',    'Ткаченко', 110000.0, 'a.tkachenko@co.ua',    1, '2020-06-20'),
    ('Сергій',  'Мельник',  65000.0,  's.melnyk@company.ua',  3, '2023-09-05'),
    ('Ірина',   'Гончар',   75000.0,  'i.gonchar@company.ua', 4, '2022-11-30'),
    ('Василь',  'Лисенко',  55000.0,  'v.lysenko@company.ua', 2, '2024-02-14'),
]

cursor.executemany(
    'INSERT INTO employees (first_name, last_name, salary, email, dept_id, hire_date) VALUES (?, ?, ?, ?, ?, ?)',
    employees_data
)

conn.commit()
print(f'Вставлено {len(departments_data)} відділів та {len(employees_data)} співробітників!')

In [ ]:
# ---- DML: UPDATE — змінюємо існуючі дані ----

# Підвищуємо зарплату конкретному співробітнику (завжди використовуємо WHERE!)
cursor.execute('''
    UPDATE employees
    SET salary = salary * 1.15,       -- підвищення на 15%
        hire_date = '2022-03-15'      -- можна оновлювати декілька полів одночасно
    WHERE email = 'o.koval@company.ua'
''')

# rowcount — кількість зачеплених рядків (корисно для перевірки)
print(f'UPDATE: змінено {cursor.rowcount} рядків')

# ---- DML: DELETE — видаляємо рядки ----

# Вставляємо тестовий запис, щоб потім видалити
cursor.execute("""
    INSERT INTO employees (first_name, last_name, salary, email, dept_id)
    VALUES ('Тест', 'Видалення', 10000.0, 'test@delete.ua', NULL)
""")
test_id = cursor.lastrowid   # lastrowid — ID щойно вставленого запису
print(f'\nВставлено тестовий запис з emp_id={test_id}')

# Видаляємо тільки цей запис — використовуємо первинний ключ для точності
cursor.execute('DELETE FROM employees WHERE emp_id = ?', (test_id,))
print(f'DELETE: видалено {cursor.rowcount} рядків (emp_id={test_id})')

conn.commit()

### DQL на практиці — SELECT

In [ ]:
# ---- DQL: SELECT — читаємо дані без змін ----

# Базовий SELECT: вибираємо конкретні колонки (не SELECT *)
print('=== Всі співробітники (сортування за зарплатою) ===')
cursor.execute('''
    SELECT
        emp_id,
        first_name || ' ' || last_name AS full_name,  -- конкатенація рядків
        salary,
        dept_id
    FROM employees
    ORDER BY salary DESC   -- сортуємо за спаданням зарплати
''')

rows = cursor.fetchall()
print(f'{"ID":<5} {"Ім'я":<20} {"Зарплата":<12} {"Відділ"}')
print('-' * 45)
for row in rows:
    dept = row['dept_id'] if row['dept_id'] else 'немає'
    print(f'{row["emp_id"]:<5} {row["full_name"]:<20} {row["salary"]:<12.1f} {dept}')

In [ ]:
# ---- DQL: WHERE, агрегатні функції, GROUP BY ----

print('=== Середня зарплата по відділах (лише відділи з серед. > 70 000) ===')
cursor.execute('''
    SELECT
        dept_id,
        COUNT(*)        AS кількість_співроб,    -- підрахунок рядків
        ROUND(AVG(salary), 2) AS середня_зарплата,  -- середнє значення
        MAX(salary)     AS максимальна,
        MIN(salary)     AS мінімальна
    FROM employees
    WHERE dept_id IS NOT NULL     -- виключаємо NULL-значення
    GROUP BY dept_id              -- групуємо результат по відділах
    HAVING AVG(salary) > 70000   -- HAVING фільтрує вже ПІСЛЯ агрегації
    ORDER BY середня_зарплата DESC
''')

rows = cursor.fetchall()
for row in rows:
    print(f'Відділ {row["dept_id"]}: '
          f'{row["кількість_співроб"]} чол., '
          f'середня={row["середня_зарплата"]:.0f} грн, '
          f'max={row["максимальна"]:.0f}, min={row["мінімальна"]:.0f}')

---
## 4. TCL — Транзакції та ACID

### ACID — чотири гарантії надійності

| Властивість | Значення | Приклад |
|-------------|----------|---------|
| **A**tomicity (Атомарність) | Або всі операції — або жодна | Переказ грошей: списали З і зарахували НА — або обидві разом |
| **C**onsistency (Узгодженість) | Транзакція веде БД з одного коректного стану в інший | Не можна видалити відділ, якщо є співробітники (foreign key) |
| **I**solation (Ізоляція) | Паралельні транзакції не "бачать" незакінчені зміни одна одної | 1000 одночасних користувачів не заважають один одному |
| **D**urability (Надійність) | Після `COMMIT` дані збережені, навіть при збої | Записи у WAL-лог перед оновленням таблиць |

### Write-Ahead Log (WAL)
Щоб не гальмувати запис на диск, БД спочатку записує зміни у **журнал (WAL-log)**, а потім асинхронно переносить у таблиці. При збої — відновлює стан, переграючи журнал.

In [ ]:
# ---- TCL: демонстрація транзакції ----
# Симуляція банківського переказу: атомарна операція (все або нічого)

# Спочатку створимо таблицю рахунків
cursor.execute('''
    CREATE TABLE IF NOT EXISTS accounts (
        acc_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        owner   TEXT    NOT NULL,
        balance REAL    NOT NULL CHECK(balance >= 0)  -- не може бути від'ємним!
    )
''')

cursor.execute("INSERT INTO accounts (owner, balance) VALUES ('Олег', 10000.0)")
cursor.execute("INSERT INTO accounts (owner, balance) VALUES ('Марія', 5000.0)")
conn.commit()

# Перевіряємо початковий стан
cursor.execute('SELECT owner, balance FROM accounts')
print('ПОЧАТКОВИЙ СТАН:')
for row in cursor.fetchall():
    print(f'  {row["owner"]}: {row["balance"]:.2f} грн')

In [ ]:
def transfer(conn, from_owner: str, to_owner: str, amount: float):
    """
    Атомарний банківський переказ.
    Або обидві операції (дебет + кредит) виконуються — або жодна.
    """
    cursor = conn.cursor()

    try:
        # BEGIN явно відкриває транзакцію
        # (в sqlite3 транзакція стартує автоматично з першою DML-командою,
        #  але явний BEGIN підкреслює намір)
        cursor.execute("BEGIN")

        # Крок 1: списуємо з рахунку відправника
        cursor.execute(
            "UPDATE accounts SET balance = balance - ? WHERE owner = ?",
            (amount, from_owner)
        )
        if cursor.rowcount == 0:
            raise ValueError(f'Рахунок {from_owner!r} не знайдено')

        # Крок 2: зараховуємо на рахунок отримувача
        cursor.execute(
            "UPDATE accounts SET balance = balance + ? WHERE owner = ?",
            (amount, to_owner)
        )
        if cursor.rowcount == 0:
            raise ValueError(f'Рахунок {to_owner!r} не знайдено')

        # Перевіряємо, що баланс відправника не від'ємний
        cursor.execute("SELECT balance FROM accounts WHERE owner = ?", (from_owner,))
        new_balance = cursor.fetchone()['balance']
        if new_balance < 0:
            raise ValueError(f'Недостатньо коштів у {from_owner!r}')

        # Все ОК — зберігаємо обидві зміни разом
        conn.commit()
        print(f'✓ Переказ {amount:.2f} грн: {from_owner} → {to_owner} — COMMIT')

    except Exception as e:
        # Щось пішло не так — скасовуємо ВСЕ з початку транзакції
        conn.rollback()
        print(f'✗ Помилка: {e} — ROLLBACK (обидва рахунки незмінні)')


# Успішний переказ
transfer(conn, 'Олег', 'Марія', 3000.0)

# Провальний переказ — недостатньо коштів
transfer(conn, 'Марія', 'Олег', 99999.0)

# Фінальний стан
cursor.execute('SELECT owner, balance FROM accounts')
print('\nФІНАЛЬНИЙ СТАН:')
for row in cursor.fetchall():
    print(f'  {row["owner"]}: {row["balance"]:.2f} грн')

In [ ]:
# ---- TCL: SAVEPOINT — часткова відкат всередині транзакції ----
# SAVEPOINT дозволяє «позначити» момент і відкотитись лише до нього,
# не скасовуючи всю транзакцію цілком

cursor.execute("BEGIN")

# Вставляємо перший запис
cursor.execute("INSERT INTO accounts (owner, balance) VALUES ('Тест1', 1000.0)")
print('Вставлено Тест1')

# Ставимо точку збереження після першого вставлення
cursor.execute("SAVEPOINT before_test2")

# Вставляємо другий запис
cursor.execute("INSERT INTO accounts (owner, balance) VALUES ('Тест2', 2000.0)")
print('Вставлено Тест2')

# "Щось пішло не так" — відкочуємось лише до savepoint (Тест1 залишиться!)
cursor.execute("ROLLBACK TO before_test2")
print('ROLLBACK TO before_test2 — Тест2 скасовано, Тест1 збережено')

# Закриваємо savepoint і комітимо (Тест1 зберігається)
cursor.execute("RELEASE before_test2")
conn.commit()

cursor.execute("SELECT owner, balance FROM accounts WHERE owner LIKE 'Тест%'")
rows = cursor.fetchall()
print(f'Результат: {[(r["owner"], r["balance"]) for r in rows]}')

# Прибираємо тестові дані
cursor.execute("DELETE FROM accounts WHERE owner LIKE 'Тест%'")
conn.commit()

---
## 5. Реляційна Архітектура

### Нормалізація

**Нормалізація** — процес розбиття даних на логічні таблиці для:
- усунення **надлишковості** (не зберігати адресу на кожному замовленні)
- забезпечення **цілісності** (змінивши адресу в одному місці — вона зміниться скрізь)

| Нормальна форма | Правило |
|----------------|--------|
| **1NF** | Всі значення атомарні (не масиви, не списки через кому) |
| **2NF** | Всі не-ключові атрибути залежать від **всього** первинного ключа |
| **3NF** | Немає транзитивних залежностей між не-ключовими атрибутами |

⚠️ **Трейдоф нормалізації:** більше таблиць → більше JOIN при читанні → нижча продуктивність без індексів.

### Типи зв'язків між таблицями

| Зв'язок | Приклад | Реалізація |
|---------|---------|------------|
| **1:M** (один до багатьох) | Відділ → Співробітники | Foreign key на стороні "багато" |
| **1:1** (один до одного) | Співробітник → Особисті дані | FK в окремій таблиці (ізоляція чутливих даних) |
| **M:N** (багато до багатьох) | Студент ↔ Курси | **Асоціативна таблиця** (junction table) з двома FK |

### Foreign Key — ключ до цілісності

Foreign Key гарантує **referential integrity**: не можна додати запис з `dept_id=999`, якщо відділу 999 не існує.

> 📌 В SQLite foreign key enforcement вимкнено за замовчуванням! Треба явно ввімкнути: `PRAGMA foreign_keys = ON`

In [ ]:
# ---- M:N зв'язок: асоціативна таблиця ----
# Приклад: проєкти та співробітники (один співробітник = багато проєктів; один проєкт = багато співроб.)

# Вмикаємо перевірку foreign keys в SQLite
cursor.execute("PRAGMA foreign_keys = ON")

# Таблиця проєктів
cursor.execute('''
    CREATE TABLE IF NOT EXISTS projects (
        proj_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        proj_name TEXT    NOT NULL,
        deadline  TEXT
    )
''')

# Асоціативна (junction) таблиця — реалізує зв'язок M:N
# Містить тільки два foreign keys і, можливо, атрибути самого зв'язку (role)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS employee_projects (
        -- Складений первинний ключ — комбінація двох FK
        emp_id  INTEGER NOT NULL REFERENCES employees(emp_id) ON DELETE CASCADE,
        proj_id INTEGER NOT NULL REFERENCES projects(proj_id) ON DELETE CASCADE,
        role    TEXT    DEFAULT 'developer',
        PRIMARY KEY (emp_id, proj_id)   -- пара унікальна: один запис = одна участь
    )
''')

# Заповнюємо проєкти
cursor.executemany(
    'INSERT INTO projects (proj_name, deadline) VALUES (?, ?)',
    [('Платіжна система', '2025-06-30'),
     ('Мобільний додаток', '2025-09-15'),
     ('BI-дашборд', '2025-04-01')]
)

# Призначаємо співробітників на проєкти
cursor.executemany(
    'INSERT INTO employee_projects (emp_id, proj_id, role) VALUES (?, ?, ?)',
    [(1, 1, 'tech lead'), (2, 1, 'developer'), (2, 2, 'developer'),
     (4, 1, 'architect'), (5, 3, 'analyst'), (3, 2, 'designer')]
)

conn.commit()
print('Таблиці projects та employee_projects створено і заповнено!')

---
## 6. JOIN Операції

### Інтуїція JOIN

Концептуально JOIN будує **декартовий добуток** (всі можливі комбінації рядків двох таблиць: M × N),
а потім фільтрує результат за умовою `ON`. Завдяки **Query Planner** реальне виконання значно ефективніше.

```
Таблиця A (M рядків) × Таблиця B (N рядків) = M×N комбінацій → фільтр ON → результат
```

### Типи JOIN

```
         ┌─────────────────────────────────────────┐
  A      │              FULL OUTER JOIN            │      B
┌────┐   │   ┌──────────────────────────────────┐  │   ┌────┐
│    │   │   │ LEFT JOIN (вся A + збіги B)      │  │   │    │
│    │   │   │   ┌──────────────────────┐       │  │   │    │
│    │   │   │   │   INNER JOIN         │       │  │   │    │
│    │   │   │   │   (перетин)          │       │  │   │    │
└────┘   │   └───┴──────────────────────┘       │  │   └────┘
         └─────────────────────────────────────────┘
```

| JOIN | Повертає |
|------|----------|
| `INNER JOIN` | Тільки рядки, де умова виконана в **обох** таблицях |
| `LEFT JOIN` | **Всі** рядки лівої + збіги правої (NULL якщо немає збігу) |
| `RIGHT JOIN` | **Всі** рядки правої + збіги лівої |
| `FULL OUTER JOIN` | Всі рядки з обох таблиць (не підтримується в SQLite) |

### Як насправді виконується JOIN (під капотом)

| Алгоритм | Коли використовується | Складність |
|----------|----------------------|------------|
| **Hash Join** | Великі несортовані таблиці | O(N + M) |
| **Nested Loop** | Одна таблиця маленька, є індекс | O(N × log M) |
| **Merge Join** | Обидві таблиці відсортовані по ключу | O(N + M) |

> 🔑 **Висновок:** завжди створюйте **індекси на Foreign Key колонках** — це змінює виконання JOIN з O(N²) на O(N log M).

In [ ]:
# ---- INNER JOIN: тільки збіги з обох таблиць ----
# Виводимо співробітників разом з назвою їхнього відділу
# Співробітники без відділу (dept_id = NULL) — НЕ потрапляють в результат

print('=== INNER JOIN: співробітники з відділами ===')
cursor.execute('''
    SELECT
        e.emp_id,
        e.first_name || ' ' || e.last_name AS full_name,
        e.salary,
        d.dept_name,
        d.location
    FROM employees e
        INNER JOIN departments d ON e.dept_id = d.dept_id
    ORDER BY d.dept_name, e.salary DESC
''')

rows = cursor.fetchall()
print(f'{"ID":<4} {"Ім'я":<22} {"Зарплата":<12} {"Відділ":<15} {"Місто"}')
print('-' * 70)
for row in rows:
    print(f'{row["emp_id"]:<4} {row["full_name"]:<22} {row["salary"]:<12.0f} '
          f'{row["dept_name"]:<15} {row["location"]}')
print(f'Рядків у результаті: {len(rows)}')

In [ ]:
# ---- LEFT JOIN: вся ліва таблиця + збіги правої ----
# Реальний use case: знайти відділи БЕЗ жодного співробітника
# (відділи, у яких немає співробітників — NULL на правій стороні)

print('=== LEFT JOIN: відділи та кількість співробітників (включно з порожніми) ===')
cursor.execute('''
    SELECT
        d.dept_id,
        d.dept_name,
        d.budget,
        COUNT(e.emp_id)       AS total_employees,  -- COUNT(NULL) = 0, не рахуємо NULL
        COALESCE(SUM(e.salary), 0) AS total_salary -- COALESCE замінює NULL на 0
    FROM departments d
        LEFT JOIN employees e ON d.dept_id = e.dept_id
    GROUP BY d.dept_id, d.dept_name, d.budget
    ORDER BY total_employees DESC
''')

rows = cursor.fetchall()
print(rows[0])
for row in rows:
    print(f'Відділ "{row["dept_name"]}" (бюджет {row["budget"]:.0f}): '
          f'{row["total_employees"]} співроб., ФОП={row["total_salary"]:.0f} грн')

In [ ]:
# ---- Три таблиці через INNER JOIN (M:N розкриття) ----
# Хто над яким проєктом працює — розкриваємо M:N через асоціативну таблицю

print('=== Співробітники та їхні проєкти (через junction table) ===')
cursor.execute('''
    SELECT
        e.first_name || ' ' || e.last_name AS employee,
        p.proj_name                        AS project,
        ep.role,
        p.deadline
    FROM employees e
        INNER JOIN employee_projects ep ON e.emp_id = ep.emp_id
        INNER JOIN projects p           ON ep.proj_id = p.proj_id
    ORDER BY employee, project
''')

rows = cursor.fetchall()
print(f'{"Співробітник":<25} {"Проєкт":<20} {"Роль":<15} {"Дедлайн"}')
print('-' * 75)
for row in rows:
    print(f'{row["employee"]:<25} {row["project"]:<20} {row["role"]:<15} {row["deadline"]}')

In [ ]:
# ---- FULL OUTER JOIN в SQLite: через UNION двох LEFT JOIN ----
# SQLite не підтримує FULL OUTER JOIN напряму.
# Стандартний обхід: LEFT JOIN + RIGHT JOIN (емульований як LEFT JOIN зі зміненим порядком)
# з'єднані через UNION

print('=== FULL OUTER JOIN (емуляція через UNION): всі відділи + всі співробітники ===')
cursor.execute('''
    -- Частина 1: всі відділи + відповідні співробітники (LEFT JOIN)
    SELECT d.dept_name, e.first_name || ' ' || e.last_name AS employee_name
    FROM departments d
        LEFT JOIN employees e ON d.dept_id = e.dept_id

    UNION

    -- Частина 2: всі співробітники + відповідні відділи
    -- (додає рядки, де співробітник є, а відділу немає — RIGHT JOIN)
    SELECT d.dept_name, e.first_name || ' ' || e.last_name AS employee_name
    FROM employees e
        LEFT JOIN departments d ON e.dept_id = d.dept_id
''')

rows = cursor.fetchall()
for row in rows:
    dept = row['dept_name'] if row['dept_name'] else '(без відділу)'
    emp  = row['employee_name'] if row['employee_name'] else '(немає співробітників)'
    print(f'  {dept:<18} | {emp}')

---
## 7. Підзапити (Subqueries) vs JOIN

**Підзапит** — запит всередині іншого запиту (у `WHERE`, `FROM`, або `SELECT`).

| Критерій | JOIN | Subquery |
|----------|------|----------|
| **Виконання** | Множинна (set-based) операція | Послідовне / вкладене |
| **Читаність** | Краще для кількох таблиць | Краще для ізольованої логіки |
| **Продуктивність** | Зазвичай швидше (оптимізатор) | Корельовані підзапити можуть бути O(N²) |
| **Колонки результату** | Доступні колонки обох таблиць | Обмежено підзапитом |

> **Практичне правило:** використовуйте JOIN, коли потрібні дані з кількох таблиць. Підзапити — для фільтрації за результатом агрегованого запиту.

In [ ]:
# ---- Підзапит у WHERE ----
# Знайти співробітників із зарплатою вищою за середню по компанії

print('=== Підзапит: співробітники з зарплатою > середньої по компанії ===')
cursor.execute('''
    SELECT
        first_name || ' ' || last_name AS full_name,
        salary,
        -- Підзапит у SELECT — обчислюється один раз для кожного рядка
        ROUND(salary - (SELECT AVG(salary) FROM employees), 2) AS above_avg
    FROM employees
    WHERE salary > (
        -- Підзапит у WHERE — виконується ОДИН раз і повертає скалярне значення
        SELECT AVG(salary) FROM employees
    )
    ORDER BY salary DESC
''')

rows = cursor.fetchall()
for row in rows:
    print(f'  {row["full_name"]:<22} {row["salary"]:.0f} грн  (+{row["above_avg"]:.0f} вище середньої)')

---
## 8. SQL як Машина Станів (концептуальний розділ)

### Схема = визначення простору станів

Схема (DDL) — це **інваріанти системи**: `NOT NULL`, `UNIQUE`, `FOREIGN KEY`, `CHECK` гарантують,
що база даних **не може перейти в невалідний стан**, незалежно від того, яку операцію ви виконуєте.

### Дані = поточний стан

Кожен рядок в таблиці — це **аксіома**: факт, який система наразі приймає за правдивий.
Коли ви вставляєте рядок — ви додаєте нову аксіому в систему знань.

### Транзакції = переходи між станами

```
Стан 0 ──BEGIN──► [незафіксований стан] ──COMMIT──► Стан 1
                          │
                       ROLLBACK
                          │
                          ▼
                        Стан 0  (повернення до попереднього стану)
```

### Запити = спостереження за станом

`SELECT` — це **читання поточного стану** без його зміни.
З точки зору реляційної алгебри, запит = **виведення нових теорем** з наявних аксіом (рядків).

### Фізична незалежність від даних

SQL відокремлює **логічну модель** (схема, запити, транзакції) від **фізичного зберігання** (індекси, сторінки диску, B-Tree структури). Додавання індексу **не змінює жодного SQL-запиту** — тільки швидкість виконання.

```
Застосунок ──SQL──► Логічна модель (що)
                           │
                    Query Planner (як)
                           │
                    Фізичне сховище (де)
```

---
## 9. Підсумок

| Концепція | Ключовий висновок |
|-----------|------------------|
| **5 підмов SQL** | DDL (структура), DML (зміна), DQL (читання), DCL (доступ), TCL (транзакції) |
| **ACID** | Атомарність + Узгодженість + Ізоляція + Надійність |
| **Нормалізація** | Мінімізує надлишковість, але вимагає JOIN при читанні |
| **Foreign Key** | Гарантує referential integrity між таблицями |
| **JOIN** | INNER (перетин), LEFT (вся ліва), FULL OUTER (все) |
| **Індекси на FK** | Критично важливі для продуктивності JOIN |
| **SQL як система** | Це state management system, а не просто мова запитів |

### Що далі?
DuckDB - аналіз даних
PostgreSQL — розширені можливості (JSON, повнотекстовий пошук, MVCC)